In [1]:
%%capture
# 1. Force uninstall torchao (the library causing the 'int1' crash)
!pip uninstall torchao -y

# 2. Install Unsloth (using the Kaggle-specific branch)
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"


In [2]:
import os
os.environ["WANDB_PROJECT"] = "off"
os.environ["WANDB_MODE"] = "disabled"

In [3]:
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset, Dataset
from trl import SFTTrainer
from transformers import TrainingArguments
import json
import os

# 1. Configuration
max_seq_length = 2048 # Supports up to 32k, but 2048 is enough for Thirukkural
dtype = None # Auto-detect (Float16 or Bfloat16)
load_in_4bit = True # 4-bit quantization to fit in free Colab/Kaggle memory

# 2. Load Model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "sarvamai/sarvam-1",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# 3. Setup LoRA (Low Rank Adaptation)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

print("✓ Model and LoRA adapters loaded successfully")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2025-12-02 07:54:41.659939: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764662082.027136      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764662082.137203      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.11.6: Fast Llama patching. Transformers: 4.57.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.77G [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

Unsloth: Will load sarvamai/sarvam-1 as a legacy tokenizer.


sarvamai/sarvam-1 does not have a padding token! Will use pad_token = <unk>.


Unsloth 2025.11.6 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


✓ Model and LoRA adapters loaded successfully


In [4]:
# 1. Load your local dataset
# UPDATE THIS PATH to where your json file is located
data_path = "/kaggle/input/i-am-kural/thirukkural_dataset_generated.json"

try:
    with open(data_path, "r", encoding="utf-8") as f:
        raw_data = json.load(f)
    print(f"✓ Loaded {len(raw_data)} examples")
except FileNotFoundError:
    # Fallback dummy data if file not found (for testing)
    print("⚠ Dataset not found. Using dummy data for demonstration.")
    raw_data = [
        {"kural": "அகர முதல எழுத்தெல்லாம் ஆதி\nபகவன் முதற்றே உலகு", 
         "generated_explanation": "Just as 'A' is the first of all letters, God is the first cause of the world."}
    ]

# 2. Define the Formatting Function
# DeepSeek-R1-Distill-Qwen uses the standard Qwen chat template
def formatting_prompts_func(examples):
    conversations = []
    
    # System prompt to give the model a persona
    sys_prompt = '''
    **"நீங்கள் ஒரு திறமையான தமிழ் அறிஞர் (Expert Tamil Scholar).
    கொடுக்கப்படும் திருக்குறள்-ஐ முழுமையாகவும் ஆழமாகவும் விளக்க வேண்டும்.
    உங்கள் விளக்கம் பின்வரும் வடிவத்தில் இருக்க வேண்டும்:**
    
    1. **தமிழ் பொருள் (Meaning in Tamil)**
    2. **ஆழமான விளக்கம் (Deep Explanation)**
    3. **கற்றுத்தரும் நன்முறை / Moral Message**
    4. **உண்மை வாழ்விலான உதாரணம் (Real-world Example)**
    
    **விளக்கங்கள் எளிமையாகவும், சரியாகவும், தவறில்லாமலும், வாசிப்பவருக்கு புரியும் முறையில் இருக்க வேண்டும்.
    '''

    for kural, explanation in zip(examples["kural"], examples["generated_explanation"]):
        # Create the message structure
        messages = [
            {"role": "system", "content": sys_prompt},
            {"role": "user", "content": f"திருக்குறள்: {kural}"},
            {"role": "assistant", "content": explanation}
        ]
        
        # Apply the chat template to turn it into a single string
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        conversations.append(text)
        
    return {"text": conversations}

# 3. Process Dataset
dataset = Dataset.from_list(raw_data)
dataset = dataset.map(formatting_prompts_func, batched=True)

print("Sample formatted data:")
print(dataset["text"][0])

✓ Loaded 2684 examples


Map:   0%|          | 0/2684 [00:00<?, ? examples/s]

Sample formatted data:
<s>[INST] <<SYS>>

    **"நீங்கள் ஒரு திறமையான தமிழ் அறிஞர் (Expert Tamil Scholar).
    கொடுக்கப்படும் திருக்குறள்-ஐ முழுமையாகவும் ஆழமாகவும் விளக்க வேண்டும்.
    உங்கள் விளக்கம் பின்வரும் வடிவத்தில் இருக்க வேண்டும்:**
    
    1. **தமிழ் பொருள் (Meaning in Tamil)**
    2. **ஆழமான விளக்கம் (Deep Explanation)**
    3. **கற்றுத்தரும் நன்முறை / Moral Message**
    4. **உண்மை வாழ்விலான உதாரணம் (Real-world Example)**
    
    **விளக்கங்கள் எளிமையாகவும், சரியாகவும், தவறில்லாமலும், வாசிப்பவருக்கு புரியும் முறையில் இருக்க வேண்டும்.
    
<</SYS>>

திருக்குறள்: எரியால் சுடப்படினும் உய்வுண்டாம் உய்யார் பெரியார்ப் பிழைத்தொழுகு வார் [/INST]
 **1. பொருள் (Meaning in Tamil)**
தீயினால் சுடப்பட்டாலும் கூட அதிலிருந்து தப்பிப் பிழைக்க ஒரு வாய்ப்பு உண்டு. ஆனால், தவத்தால் பெரியவர்களையும், ஞானத்தால் உயர்ந்தவர்களையும் அவமதித்து நடப்பவர்கள் ஒருபோதும் தப்பிப் பிழைக்க முடியாது.

**2. ஆழமான விளக்கம் (Deep Explanation)**
இந்தக் குறள், உடல் ரீதியான துன்பங்களை விட, ஒழுக்கம் மற்றும் ஆன்மிக ரீதி

In [5]:
print("\n" + "="*80)
print("TRAIN/VALIDATION SPLIT (IMPROVED)")
print("="*80)

train_val_split = dataset.train_test_split(test_size=0.1, seed=3407, shuffle=True)
train_dataset = train_val_split["train"]
eval_dataset = train_val_split["test"]

print(f"✓ Training samples: {len(train_dataset)}")
print(f"✓ Validation samples: {len(eval_dataset)}")



TRAIN/VALIDATION SPLIT (IMPROVED)
✓ Training samples: 2415
✓ Validation samples: 269


In [6]:
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

training_args = TrainingArguments(
    per_device_train_batch_size = 4,
    per_device_eval_batch_size = 4,
    gradient_accumulation_steps = 2,

    warmup_ratio = 0.05,
    num_train_epochs = 3,            # Better than max_steps
    learning_rate = 1e-4,            # LOWER LR to avoid memorization
    weight_decay = 0.05,             # Regularization

    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),

    logging_steps = 10,
    eval_steps = 25,                 # More frequent eval → faster early stop
    save_steps = 25,

    eval_strategy = "steps",
    save_strategy = "steps",

    load_best_model_at_end = True,
    metric_for_best_model = "eval_loss",
    greater_is_better = False,

    optim = "adamw_8bit",
    seed = 3407,
    report_to = "none",
    output_dir = "outputs",
    save_total_limit = 2,
)


In [7]:
from transformers import TrainingArguments
from trl import SFTTrainer
import os
import torch
import json
import matplotlib.pyplot as plt
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import TrainingArguments, EarlyStoppingCallback
from trl import SFTTrainer
from unsloth import FastLanguageModel
callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]

In [8]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    args=training_args,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/2415 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/269 [00:00<?, ? examples/s]

In [9]:
import types
# ==========================================================
def patched_compute_loss(self, model, inputs, return_outputs=False, **kwargs):
    """
    A patched version of compute_loss that removes the 'num_items_in_batch' 
    argument if present, preventing crashes with newer Transformers versions.
    """
    if "num_items_in_batch" in kwargs:
        del kwargs["num_items_in_batch"]
    
    # Call the original SFTTrainer compute_loss
    return SFTTrainer.compute_loss(self, model, inputs, return_outputs=return_outputs, **kwargs)

# Apply the patch to your trainer instance
trainer.compute_loss = types.MethodType(patched_compute_loss, trainer)

print("✓ Trainer loss function patched successfully.")

# ==========================================================
# NOW RUN TRAINING
# ==========================================================
trainer_stats = trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.


✓ Trainer loss function patched successfully.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,415 | Num Epochs = 3 | Total steps = 453
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 2 x 1) = 16
 "-____-"     Trainable parameters = 23,969,792 of 2,549,057,536 (0.94% trained)
Unsloth: Not an error, but LlamaForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
25,4.688300,2.058538
50,3.402600,1.679355
75,3.256300,1.602893
100,3.116600,1.562993
125,3.076400,1.537253
150,3.009900,1.518201
175,2.996600,1.504044
200,2.946600,1.491389
225,2.932400,1.481972
250,2.937900,1.473444


In [11]:
repo_name = "sadie26032005/sarvam-ai-finetuning-distillation-v4-1"



# Replace with your repo name

model.push_to_hub(repo_name,safe_serialization=False)


tokenizer.push_to_hub(repo_name)

README.md:   0%|          | 0.00/560 [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved model to https://huggingface.co/sadie26032005/sarvam-ai-finetuning-distillation-v4-1


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            